In [ ]:
# program arguments
YEAR = "2024"
MONTH = "1"

IGNORE_MISSING = False

In [ ]:
# program variables
MONTH_FMT = MONTH.zfill(2)

FILE_PREFIX = f"{YEAR}{MONTH_FMT}"
FILE_NAME = f"{FILE_PREFIX}-citibike-tripdata.zip"
FILE_URL = f"https://s3.amazonaws.com/tripdata/{FILE_NAME}"


BUCKET_NAME = "de_citibike_bucket"
GCS_FILE_URL = f"{YEAR}/{FILE_NAME}"

CHUNK_SIZE_BYTES = 1024 * 1024 # 1MB

In [ ]:
# from typing import Union

# import requests
# from requests import Response


# def _validate_file_contents(r: Response) -> Union[Response, None]:
#     """Validates a http response for file existence.

#     Returns the requests Response object if file contents not empty.
#     If IGNORE_MISSING is True, skip current file and return None.

#     Raises exception if file is missing and IGNORE_MISSING is False.
#     """

#     print("[INFO] Validating remote file and file contents.")

#     if r.headers["content-length"] and int(r.headers["content-length"]) > 0:
#         return r

#     if IGNORE_MISSING:
#         print("f[INFO] The remote file is missing or empty. Skip download.")
#         return None
#     else:
#         raise ValueError(f"Remote file is missing or empty.")

In [ ]:
# setup env vars
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()

# generate path to gcp account api keys json file
ROOT = Path().resolve().parents[0]
gcp_cred_path = ROOT / os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(gcp_cred_path)

In [ ]:
# from google.cloud import storage
# from google.cloud.exceptions import Conflict

# # TODO - remove bucket_name, re-use BUCKET_NAME (when dev done)
# # bucket_name = "de_citibike_bucket"
# bucket_name = "de_course_bucket"

# storage_client = storage.Client()

# # sample GS url: gs://de_course_bucket/taxi/green_tripdata_2019-04.csv


# def _file_exists_in_storage(
#     file_name: str, client: storage.Client
# ) -> Union[storage.Bucket, storage.Blob]:
#     """Verifies if the file exists in the google storage.

#     Returns the bucket object if file is missing (for further file creation).
#     Returns the Blob containing the file.
#     """

#     # bucket = client.get_bucket(BUCKET_NAME)
#     bucket = client.get_bucket(bucket_name)

#     blobs = bucket.list_blobs()

#     for blob in blobs:
#         if blob.name == file_name:
#             print(f"blob.size: {blob.size}")
#             print(blob.md5_hash)
#             return blob

#     return bucket


# # def _file_exists_in_storage(file_name: str, client: storage.Client) -> bool:
# #     return True


# # remote file size: 369035302
# blob = _file_exists_in_storage("taxi/green_tripdata_2019-04.csv", storage_client)

# for k, v in vars(blob).items():
#     if k == "_properties":
#         for kk, vv in v.items():
#             print(f"prop: {kk}:{vv}")
#     print(k, v)
# # _file_exists_in_storage(GCS_FILE_URL, storage_client)

In [ ]:
# # Pseudo Code

# # NOTE: assume bucket exists. It's managed by terraform

# # check file exists on internet
# # - yes -> next
# # - no -> skip -> log
# # check file exists in GCS
# # - yes -> check file checksum (crc32c - preferable, md5 - also good)
# # - -   checksum the same -> skip -> log
# # - -   checksum different -> upload to GCS (replace) -> log
# # - no -> upload to GCS
# # done

# def ingest(from_url: str, to_bucket: str):
#     """Extract dataset files from_url and upload these to a cloud storage.

#     First it checks file exists at the given from_url.
#     If success, next checks if the same file exists in the cloud storage.
#     It doesn't load and existing file twice to the same storage.
#     Finally it upload the file to the storage if this is new.
#     """

#     with requests.get(from_url, stream=True) as r:
#         r.raise_for_status()
#         for (k, v) in r.headers.items():
#             print(f"k: {k} - v: {v}")

#         http_response = _validate_file_contents(r)
#         # crc_hash = None
#         # storage_file = _file_exists_in_storage(GCS_FILE_URL, storage_client)
#         if http_response:
#             pass
#             # for line in r.iter_content(chunk_size=CHUNK_SIZE_BYTES):
#             #     pass


# ingest(FILE_URL, "blah")

In [ ]:
# def ingest(from_url: str, to_bucket: str):
#     """
#     Download a file from a remote URL and upload it to GCS
#     only if it does not exist or differs in size.

#     Comparison strategy:
#         - Use HTTP Content-Length (remote size) when available
#         - Compare with GCS blob.size
#         - If Content-Length is missing → force download
#     """

#     print(f"[INFO] Starting ingest: {from_url}")

#     client = storage.Client()
#     bucket = client.bucket(to_bucket)
#     blob_name = GCS_FILE_URL

#     # Step 1: Fetch metadata only (no file download)
#     r = requests.head(from_url, allow_redirects=True)
#     r.raise_for_status()

#     r = _validate_file_contents(r)
#     if r is None:
#         return

#     content_length = r.headers.get("content-length")

#     # Step 2: Check existing file in GCS
#     blob = bucket.blob(blob_name)
#     blob_exists = blob.exists()

#     if content_length is None:
#         print("[INFO] Content-Length missing. Forcing download.")

#     elif blob_exists:
#         blob.reload()
#         remote_size = int(content_length)
#         gcs_size = blob.size

#         print(f"[INFO] Remote size: {remote_size}, GCS size: {gcs_size}")

#         if gcs_size == remote_size:
#             print("[INFO] File already exists with same size. Skipping.")
#             return
#         else:
#             print("[INFO] Size mismatch. Re-uploading file.")

#     else:
#         print("[INFO] File not found in GCS. Uploading.")

#     # Step 3: Download and upload (single pass)
#     with requests.get(from_url, stream=True) as r:
#         r.raise_for_status()

#         blob = bucket.blob(blob_name)

#         with blob.open("wb") as f:
#             for chunk in r.iter_content(chunk_size=CHUNK_SIZE_BYTES):
#                 if chunk:
#                     f.write(chunk)

#     print(f"[INFO] Upload complete: gs://{to_bucket}/{blob_name}")

In [ ]:
# import os
# import tempfile
# import requests
# from google.cloud import storage
# from google.api_core.retry import Retry

# CHUNK_SIZE = 8 * 1024 * 1024  # 8MB
# REQUEST_TIMEOUT = (10, 300)   # (connect, read)

# RETRY = Retry(
#     initial=1.0,
#     maximum=60.0,
#     multiplier=2.0,
#     deadline=300.0,
# )


# def ingest(from_url: str, to_bucket: str, blob_name: str) -> None:
#     client = storage.Client()
#     bucket = client.bucket(to_bucket)
#     blob = bucket.blob(blob_name)

#     # ---- Step 1: HEAD (safe timeout)
#     head = requests.head(from_url, allow_redirects=True, timeout=REQUEST_TIMEOUT)
#     head.raise_for_status()

#     content_length = head.headers.get("content-length")
#     remote_size = int(content_length) if content_length else None

#     # ---- Step 2: GCS check
#     if blob.exists():
#         blob.reload()
#         if remote_size and blob.size == remote_size:
#             print("[INFO] Skip: already exists with same size")
#             return

#     # ---- Step 3: Download to temp file (robust)
#     with tempfile.NamedTemporaryFile(delete=False) as tmp:
#         tmp_path = tmp.name

#         with requests.get(from_url, stream=True, timeout=REQUEST_TIMEOUT) as r:
#             r.raise_for_status()

#             for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
#                 if chunk:
#                     tmp.write(chunk)

#     # ---- Step 4: Upload (resumable, retryable)
#     blob.chunk_size = CHUNK_SIZE  # align with download

#     with open(tmp_path, "rb") as f:
#         blob.upload_from_file(
#             f,
#             rewind=True,
#             timeout=300,
#             retry=RETRY,
#         )

#     os.remove(tmp_path)

#     print(f"[INFO] Uploaded: gs://{to_bucket}/{blob_name}")

In [ ]:
# ingest(FILE_URL, BUCKET_NAME, GCS_FILE_URL)

In [ ]:
# ingest(FILE_URL, BUCKET_NAME, GCS_FILE_URL)